# 6 · A person in the loop

A node that needs a person raises `Ask`, and its `@node` says it may, with `asks=True`. The node's return type says what the answer is. Only the calls that need the answer wait. The rest of the graph runs on, and the run ends `waiting`, with each question in `run.waiting`. `run.with_answers(waiting, value)` gives an answer, and `run.resume()` goes on.

In [ ]:
from typing import Annotated

from conductor import Ask, Graph, InvalidCallError, NodeRegistry, Param, Series
from conductor.widgets import Textarea

registry = NodeRegistry()


@registry.node("text-uppercase", title="Uppercase", description="Turns a text into capital letters.", category="text")
def uppercase(text: str) -> str:
    return text.upper()


@registry.node("text-split", title="Split", description="Splits a text at a separator.", category="text")
def split(text: str, separator: str = ",") -> Series[str]:
    return text.split(separator)


@registry.node("text-join", title="Join", description="Joins the pieces into one text.", category="text")
def join(parts: Series[str], separator: str = "") -> str:
    return separator.join(parts)


@registry.node("approve", title="Approve", description="Asks a person to approve the proposal.", category="review", asks=True)
def approve(proposal: str) -> Annotated[str, Param(title="Decision", widget=Textarea())]:
    raise Ask("Approve or rewrite the proposal.", default=proposal)

## One question

`shout` needs `approve`'s answer, so it waits. `other` needs nothing from it, so it runs in the same `run()`.

Each entry of `run.waiting` has an `id`, the `prompt`, and `questions`, one per output of the node. A question is the output as `describe()` shows it, with what the `Ask` added, such as its `default`. A host draws a form from it as it draws an input.

In [ ]:
graph = Graph.compile({
    "nodes": {
        "approve": {"node": "approve", "inputs": {"proposal": {"constant": "Ship it"}}},
        "shout": {"node": "text-uppercase", "inputs": {"text": {"sources": [{"node": "approve", "output": "result"}]}}},
        "other": {"node": "text-uppercase", "inputs": {"text": {"constant": "independent"}}},
    },
    "outputs": {"shout": {"node": "shout", "output": "result"}, "other": {"node": "other", "output": "result"}},
}, registry)

run = graph.run()

print(run.status)
for waiting in run.waiting:
    print(waiting.id, waiting.prompt)
    print([(question.name, question.type, question.default) for question in waiting.questions])
print("finished:", {node_id: count.finished for node_id, count in run.progress.nodes.items()})

## Answer, and resume

A run never changes in place: `with_answers` and `resume` each return a new run, so keep what they return. The answer is checked before anything runs. `resume()` calls only what is left, here `shout`.

In [ ]:
done = run.with_answers(run.waiting[0], "Ship it on Monday").resume()

print(done.status)
print(done.outputs)
print("finished:", {node_id: count.finished for node_id, count in done.progress.nodes.items()})

## What was asked, and declining

`run.asked` keeps every question a person answered or declined: the question as it was shown, and the answer. It is saved with the run, for the host's records.

To stop without answering, `run.with_decline(waiting, reason=...)` ends the run for good: it is `declined`, and nothing more is called. To turn one item down and let the run go on, answer it instead.

In [ ]:
for asked in done.asked:
    print(asked.question.prompt, "->", asked.answer)

declined = run.with_decline(run.waiting[0], reason="Not this week")
print(declined.status, declined.cause.ctx if declined.cause else None)

## A question per item

A node that runs once per item asks once per item. Each waiting call has its own id: the node, its item index, and a fingerprint of the node and the inputs it was called with (`approve@1~3fa2c0d1`). `run.with_answers({id: value, ...})` answers several at once, and refuses every wrong one together. A call left out goes on waiting.

In [ ]:
graph = Graph.compile({
    "nodes": {
        "drafts": {"node": "text-split", "inputs": {"text": {"constant": "first draft,second draft"}}},
        "approve": {"node": "approve", "inputs": {"proposal": {"sources": [{"node": "drafts", "output": "result"}]}}},
        "joined": {"node": "text-join", "inputs": {
            "parts": {"sources": [{"node": "approve", "output": "result"}]},
            "separator": {"constant": " | "},
        }},
    },
    "outputs": {"joined": {"node": "joined", "output": "result"}},
}, registry)

run = graph.run()
print(run.status, [(waiting.id, waiting.questions[0].default) for waiting in run.waiting])

run = run.with_answers({waiting.id: f"{waiting.questions[0].default}, approved" for waiting in run.waiting})

done = run.resume()
print(done.status, done.outputs)

Between calls of `resume()` a run is JSON, so it can wait for days in a host's database, and be answered in another process:

```python
saved = run.to_json()
run = Run.from_json(saved, graph)  # in another process, with the same graph compiled again
```

## How the return type asks

The return type says what is asked: `-> str` a line of text, `-> bool` a yes or no, and a dataclass a form, with one question per field. `python -m conductor.about asking-a-person` lists every type.

- Fixed choices are part of the type, as on `labels` below, so `describe()` shows them before any run.
- Choices that are data, such as a team that comes in on an input, are offered with `Ask(choices=...)`, as `owner` does.
- In a form, a field's own default is proposed, and an `Ask`'s `default=` and `choices=` go by output name, as the answer does. How each field is asked, its title and its widget, is its `Param`.

A question with no default has `MISSING` there, pydantic's mark for a key left out of its JSON. `None` is a default of `None`.

In [ ]:
import datetime
from dataclasses import dataclass


@dataclass(frozen=True)
class Review:
    approved: bool
    note: Annotated[str | None, Param(widget=Textarea(rows=3))]
    due: datetime.date = datetime.date(2026, 12, 1)


@registry.node("review", title="Review", description="Asks a person to review a proposal.", asks=True)
def review(proposal: str) -> Review:
    raise Ask(f"Review: {proposal}")


@registry.node("labels", title="Labels", description="Asks a person which labels a proposal gets.", asks=True)
def labels(proposal: str) -> Series[Annotated[str, Param(choices=["urgent", "legal", "money"])]]:
    raise Ask("Which labels?", default=["urgent"])


@registry.node("owner", title="Owner", description="Asks a person who owns a proposal.", asks=True)
def owner(proposal: str, team: list[str]) -> str:
    raise Ask("Who owns it?", choices=team)


graph = Graph.compile({
    "nodes": {
        "r": {"node": "review", "inputs": {"proposal": {"constant": "Ship it"}}},
        "l": {"node": "labels", "inputs": {"proposal": {"constant": "Ship it"}}},
        "o": {"node": "owner", "inputs": {"proposal": {"constant": "Ship it"}, "team": {"constant": ["ida", "bo"]}}},
    },
    "outputs": {
        "approved": {"node": "r", "output": "approved"},
        "note": {"node": "r", "output": "note"},
        "labels": {"node": "l", "output": "result"},
        "owner": {"node": "o", "output": "result"},
    },
}, registry)
run = graph.run()
for waiting in run.waiting:
    print(waiting.id, waiting.prompt)
    for question in waiting.questions:
        print("   ", question.name, question.required, question.default, question.choices)

A form is answered by output name. An output with a default may be left out, and so may one whose type takes `None` and that has no choices. A pick of several is a list of the choices. Every wrong answer is refused at once:

In [ ]:
try:
    run.with_answers({"r": {"note": "fine"}, "l": ["urgent", "urgent"], "o": "carl"})
except InvalidCallError as wrong:
    for problem in wrong.problems:
        print(problem.loc, problem.msg)

done = run.with_answers({"r": {"approved": True}, "l": ["urgent", "money"], "o": "ida"}).resume()
print(done.status, done.outputs)